# Experiment: GP-KGE (Ours)

**Loss:** BCE
**Kernel:** Relation-Aware
**Seeds:** 42, 123, 456 (3 runs)

In [1]:
# Setup
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !rm -rf /content/kg-bayesian-prior
    !git clone https://github.com/ChorokLeeDev/kg-bayesian-prior.git /content/kg-bayesian-prior
    !pip install -q torch-geometric gpytorch networkx pandas tqdm scikit-learn
    os.chdir('/content/kg-bayesian-prior')
    sys.path.insert(0, '/content/kg-bayesian-prior')
else:
    sys.path.insert(0, os.path.dirname(os.getcwd()))

Cloning into '/content/kg-bayesian-prior'...
remote: Enumerating objects: 217, done.
remote: Counting objects: 100% (217/217), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 217 (delta 126), reused 154 (delta 63), pack-reused 0 (from 0)
Receiving objects: 100% (217/217), 184.83 KiB | 844.00 KiB/s, done.
Resolving deltas: 100% (126/126), done.


In [2]:
import gc, json, warnings
import torch
import torch.nn.functional as F
import numpy as np
from scipy import sparse
from scipy.sparse.linalg import eigsh
from tqdm.notebook import tqdm

from src.data import load_fb15k237
from src.models import GPKGE
from src.kernels.matern_graph import GraphLaplacian
from src.utils.training import set_seed
from src.evaluation.calibration import expected_calibration_error, brier_score
from src.evaluation.ood_detection import compute_auroc, create_ood_dataset

warnings.filterwarnings('ignore')
device = "cuda" if torch.cuda.is_available() else "cpu"

# GPU info
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU available")

train_data, _, test_data = load_fb15k237()
print(f"Data: {len(train_data):,} train, {len(test_data):,} test")

GPU: NVIDIA A100-SXM4-80GB
Memory: 85.2 GB
FB15k-237 not found. Downloading...


train.txt: 21.0MB [00:01, 11.9MB/s]


valid.txt: 1.29MB [00:00, 2.90MB/s]


test.txt: 1.51MB [00:00, 4.79MB/s]


FB15k-237 download complete!
Data: 272,115 train, 20,466 test


In [ ]:
def setup_eigendecomp(model):
    """Setup eigendecomposition for relation-aware kernel"""
    kernel = model.kernel
    kernel.num_entities = train_data.num_entities
    kernel.relation_laplacians = {}

    success, failed = 0, 0
    for rel_id, adj in tqdm(train_data.relation_adjacencies.items(), desc="Eigendecomp", leave=False):
        if adj.nnz < 10:
            continue
        try:
            degrees = np.array(adj.sum(axis=1)).flatten()
            D_inv_sqrt = sparse.diags(1.0 / np.sqrt(np.maximum(degrees, 1e-10)))
            L = sparse.diags(degrees) - adj
            L_norm = D_inv_sqrt @ L @ D_inv_sqrt
            L_norm = (L_norm + L_norm.T) / 2
            k = min(100, L_norm.shape[0] - 2)
            if k < 2:
                continue
            eigvals, eigvecs = eigsh(L_norm, k=k, which='SM', maxiter=1000, tol=1e-4)
            kernel.relation_laplacians[rel_id] = GraphLaplacian(adj.shape[0])
            kernel.relation_laplacians[rel_id].eigenvalues = torch.tensor(eigvals, dtype=torch.float32)
            kernel.relation_laplacians[rel_id].eigenvectors = torch.tensor(eigvecs, dtype=torch.float32)
            success += 1
        except:
            failed += 1
    print(f"Eigendecomp: {success} success, {failed} failed")

In [4]:
def train_and_evaluate(seed):
    print(f"\n{'='*50}")
    print(f"SEED {seed}")
    print(f"{'='*50}")
    set_seed(seed)
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    # Model setup
    model = GPKGE(
        train_data.num_entities,
        train_data.num_relations,
        embedding_dim=200,
        kernel_type="relation_aware",
        num_inducing=500
    ).to(device)

    setup_eigendecomp(model)

    # Training
    opt = torch.optim.Adam(model.parameters(), lr=0.001)

    for ep in (pbar := tqdm(range(50), desc=f"GP-KGE (seed={seed})")):
        model.train()
        loss_sum, n = 0, 0
        for st in range(0, len(train_data), 1024):
            pos = torch.tensor(train_data.triples[st:st+1024], device=device)
            neg = pos.clone()
            neg[:,2] = torch.randint(0, train_data.num_entities, (len(pos),), device=device)
            opt.zero_grad()
            ps = model.score_triple(pos[:,0], pos[:,1], pos[:,2], use_mean=True)
            ns = model.score_triple(neg[:,0], neg[:,1], neg[:,2], use_mean=True)
            loss = F.binary_cross_entropy_with_logits(
                torch.cat([ps, ns]), torch.cat([torch.ones_like(ps), torch.zeros_like(ns)]))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            loss_sum += loss.item()
            n += 1
        pbar.set_postfix(loss=f"{loss_sum/n:.4f}")

    # Evaluation
    model.eval()

    # MRR
    ranks = []
    with torch.no_grad():
        for i in tqdm(range(0, len(test_data), 200), desc="MRR", leave=False):
            batch = test_data.triples[i:i+200]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
            scores = model.score_tails(h, r)
            target = scores[torch.arange(len(t), device=device), t]
            ranks.extend(((scores > target.unsqueeze(1)).sum(1) + 1).cpu().tolist())

    ranks = torch.tensor(ranks, dtype=torch.float)
    mrr = (1/ranks).mean().item()
    h1 = (ranks <= 1).float().mean().item()
    h10 = (ranks <= 10).float().mean().item()

    # ECE
    pos = test_data.triples
    neg = np.array([[h, r, np.random.randint(train_data.num_entities)] for h,r,t in pos])
    all_t = np.vstack([pos, neg])
    labels = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])

    confs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(all_t), 1024), desc="ECE", leave=False):
            batch = all_t[i:i+1024]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
            scores = model.score_triple(h, r, t)
            confs.append(torch.sigmoid(scores).cpu().numpy())
    conf = np.concatenate(confs)
    ece, _ = expected_calibration_error(conf, labels)
    brier = brier_score(conf, labels)

    # AUROC
    ood_t = create_ood_dataset(train_data, test_data, "random", len(test_data))

    def get_unc(triples):
        uncs = []
        with torch.no_grad():
            for i in range(0, len(triples), 1024):
                batch = triples[i:i+1024]
                h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
                pred = model.predict_with_uncertainty(h, r, t)
                uncs.append(pred['total'].cpu().numpy())
        return np.concatenate(uncs)

    auroc = compute_auroc(get_unc(test_data.triples), get_unc(ood_t))

    result = {"mrr": mrr, "hits@1": h1, "hits@10": h10, "ece": ece, "brier": brier, "auroc": auroc}
    print(f"Result: MRR={mrr:.4f}, H@1={h1:.4f}, H@10={h10:.4f}, ECE={ece:.4f}, AUROC={auroc:.4f}")

    del model
    return result

In [ ]:
# Run multiple seeds
SEEDS = [42, 123, 456]
all_results = {}

for seed in SEEDS:
    all_results[seed] = train_and_evaluate(seed)


SEED 42


Eigendecomp:   0%|          | 0/237 [00:00<?, ?it/s]

Eigendecomp: 223 success, 14 failed


GP-KGE (seed=42):   0%|          | 0/50 [00:00<?, ?it/s]

MRR:   0%|          | 0/103 [00:00<?, ?it/s]

...

GP-KGE (seed=456):   0%|          | 0/50 [00:00<?, ?it/s]

MRR:   0%|          | 0/103 [00:00<?, ?it/s]

ECE:   0%|          | 0/40 [00:00<?, ?it/s]

Result: MRR=0.2534, H@1=0.1751, H@10=0.4137, ECE=0.1192, AUROC=0.8546


: 

In [6]:
# Aggregate results
metrics = ['mrr', 'hits@1', 'hits@10', 'ece', 'brier', 'auroc']

print("\n" + "="*70)
print("GP-KGE RESULTS (mean ± std)")
print("="*70)

summary = {}
for m in metrics:
    values = [all_results[s][m] for s in SEEDS]
    mean, std = np.mean(values), np.std(values)
    summary[m] = {"mean": mean, "std": std, "values": values}
    print(f"{m:>10}: {mean:.4f} ± {std:.4f}")

# Save to results folder
output = {"seeds": SEEDS, "results": all_results, "summary": summary}
result_path = "results/gpkge_results.json"
os.makedirs("results", exist_ok=True)
with open(result_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f"\nSaved to {result_path}")

# Push to GitHub and shutdown (Colab only)
if IN_COLAB:
    !git config user.email "colab@experiment.com"
    !git config user.name "Colab Experiment"
    !git add results/gpkge_results.json
    !git commit -m "Add GP-KGE experiment results"
    !git push
    print("Pushed to GitHub!")
    print("Shutting down kernel...")
    import os
    os._exit(0)


GP-KGE RESULTS (mean ± std)
       mrr: 0.2550 ± 0.0011
    hits@1: 0.1777 ± 0.0022
   hits@10: 0.4130 ± 0.0005
       ece: 0.1181 ± 0.0008
     brier: 0.1025 ± 0.0006
     auroc: 0.8542 ± 0.0025

Saved to results/gpkge_results.json
[main ad0aa5c] Add GP-KGE experiment results
 1 file changed, 89 insertions(+)
 create mode 100644 results/gpkge_results.json
fatal: could not read Username for 'https://github.com': No such device or address


: 

: 